# OpenAQ Real-Time Air Quality Dataset Collector
### For: *Multi-Station IoT Sensor Dataset for Anomaly Detection in Autonomic Monitoring Loops*

**Before running:** Paste your API key in Cell 2.  
Get one free at → **https://openaq.org/**
check out README.txt for further instructions

Run cells **top to bottom**, one at a time.

In [50]:
# 1 — Install libraries
!pip install requests pandas --quiet
print('✅ Libraries ready.')

✅ Libraries ready.


In [51]:
# 2 — Paste your API key
API_KEY = 'YOUR_API_KEY_HERE'   # <── Replace this with your key

if API_KEY == 'YOUR_API_KEY_HERE': ## note for anyone starting out, the api only goes on the first one (:
    raise ValueError(' Please paste your OpenAQ API key above!')

import requests, pandas as pd, numpy as np, time
from datetime import datetime, timedelta, timezone

BASE_URL = 'https://api.openaq.org/v3'
HEADERS  = {'X-API-Key': API_KEY, 'Accept': 'application/json'}

# ── Live connection test ──
r = requests.get(f'{BASE_URL}/parameters?limit=1', headers=HEADERS, timeout=10)
if r.status_code == 200:
    print('✅ API key works! Connected to OpenAQ v3.')
elif r.status_code == 401:
    raise ValueError('⛔ API key rejected — double-check you copied it fully.')
else:
    print(f'⚠️  Status {r.status_code}: {r.text[:300]}')

✅ API key works! Connected to OpenAQ v3.


In [52]:
#  3 — Look up numeric country IDs from the API
#
# The OpenAQ v3 API requires numeric country IDs, NOT ISO codes.
## This cell fetches the full countries list and builds a lookup table.

TARGET_ISO_CODES = ['MX'] ## Change this to look at another country, like US = United States, BR = Brasil, etc.

def get_all_countries():
    countries = []
    page = 1
    while True:
        r = requests.get(
            f'{BASE_URL}/countries',
            headers=HEADERS,
            params={'limit': 200, 'page': page},
            timeout=15
        )
        if r.status_code != 200:
            break
        data = r.json()
        results = data.get('results', [])
        if not results:
            break
        countries.extend(results)
        if len(results) < 200:
            break
        page += 1
    return countries

print('🔍 Fetching country list from OpenAQ...')
all_countries = get_all_countries()
print(f'   Found {len(all_countries)} countries total in OpenAQ.')

# Build iso_code → numeric_id map
country_id_map = {}
for c in all_countries:
    iso  = c.get('code', '')
    cid  = c.get('id')
    name = c.get('name', '')
    if iso in TARGET_ISO_CODES:
        country_id_map[iso] = {'id': cid, 'name': name}

print('\n✅ Numeric IDs resolved for target countries:')
for iso, info in country_id_map.items():
    print(f'   {iso}  →  ID {info["id"]:>4}  ({info["name"]})')

missing = [iso for iso in TARGET_ISO_CODES if iso not in country_id_map]
if missing:
    print(f'\n⚠️  Not found in OpenAQ: {missing} (they may have no data)')

🔍 Fetching country list from OpenAQ...
   Found 150 countries total in OpenAQ.

✅ Numeric IDs resolved for target countries:
   MX  →  ID  157  (Mexico)


In [63]:
# 4 - Bounding boxes for each Mexico City quadrant
# Format: (name, lat_min, lat_max, lon_min, lon_max, n_stations)
## For more details about the cuadrants please check out README.txt
QUADRANTS = [
    ('NW_Industrial',   19.45, 19.70, -99.40, -99.05, 5),
    ('NE_Transport',    19.45, 19.70, -99.05, -98.80, 4),
    ('SW_UrbanCore',    19.20, 19.50, -99.40, -99.05, 5),
    ('SE_Background',   19.20, 19.50, -99.05, -98.80, 4),
] ### if you want individual sensor points you may do so by changing the function above.

def discover_stations_bbox(lat_min, lat_max, lon_min, lon_max, n=3):
    r = requests.get(
        f'{BASE_URL}/locations',
        headers=HEADERS,
        params={
            'bbox': f'{lon_min},{lat_min},{lon_max},{lat_max}',
            'limit': 50,
            'order_by': 'id',
        },
        timeout=15
    )
    if r.status_code != 200:
        print(f'      API error {r.status_code}: {r.text[:100]}')
        return []
    return r.json().get('results', [])[:n]

print('Discovering stations by quadrant...\n')
STATIONS = []

for quad_name, lat_min, lat_max, lon_min, lon_max, n_want in QUADRANTS:
    found = discover_stations_bbox(lat_min, lat_max, lon_min, lon_max, n_want)
    print(f'  {quad_name}: {len(found)} stations found')
    for loc in found:
        coords = loc.get('coordinates', {})
        STATIONS.append({
            'id':          loc['id'],
            'name':        loc.get('name', 'Unknown'),
            'city':        'Mexico City',
            'quadrant':    quad_name,
            'country':     'MX',
            'latitude':    coords.get('latitude'),
            'longitude':   coords.get('longitude'),
            'sensor_type': 'government' if loc.get('isMonitor') else 'low-cost',
        })
        print(f'    [{loc["id"]:>7}] {loc.get("name","")[:45]}')
    time.sleep(3.0)

print(f'\n✅ Ready: {len(STATIONS)} stations across 4 quadrants.')

Discovering stations by quadrant...

  NW_Industrial: 5 stations found
    [    759] Atizapán
    [    918] Tultitlán
    [   1271] La Presa
    [   1367] Xalostoc
    [   1505] Villa de las Flores
  NE_Transport: 4 stations found
    [   1237] Acolman
    [   1578] Los Laureles
    [   1830] Montecillo
    [   2020] San Agustín
  SW_UrbanCore: 5 stations found
    [    348] Santa Fe
    [    576] Pedregal
    [    926] UAM Xochimilco
    [   1090] Iztacalco
    [   1134] Hospital General de México
  SE_Background: 4 stations found
    [    677] Tlahuac
    [   1809] Chalco
    [   1830] Montecillo
    [   2054] Nezahualcóyotl

✅ Ready: 18 stations across 4 quadrants.


In [64]:
# Cell 4b — validate stations
## a note, some sensors have are not entirely up to date, this cell has been added to verify their status.
import time

def has_sensors(loc_id):
    r = requests.get(
        f'{BASE_URL}/locations/{loc_id}/sensors',
        headers=HEADERS, timeout=10
    )
    if r.status_code != 200:
        return False
    sensors = r.json().get('results', [])
    criteria = {'pm25','pm10','no2','o3','co','so2'}
    return any(
        s.get('parameter', {}).get('name', '').lower() in criteria
        for s in sensors
    )

print('🔍 Pre-validating stations...\n')
VALID_STATIONS = []
for st in STATIONS:
    ok = has_sensors(st['id'])
    status = '✅' if ok else '❌'
    print(f'  {status}  [{st["id"]:>7}]  {st["name"][:45]}')
    if ok:
        VALID_STATIONS.append(st)
    time.sleep(0.3)

print(f'\n✅ {len(VALID_STATIONS)} stations confirmed.')
STATIONS = VALID_STATIONS

🔍 Pre-validating stations...

  ✅  [    759]  Atizapán
  ✅  [    918]  Tultitlán
  ✅  [   1271]  La Presa
  ✅  [   1367]  Xalostoc
  ✅  [   1505]  Villa de las Flores
  ✅  [   1237]  Acolman
  ✅  [   1578]  Los Laureles
  ✅  [   1830]  Montecillo
  ✅  [   2020]  San Agustín
  ✅  [    348]  Santa Fe
  ✅  [    576]  Pedregal
  ✅  [    926]  UAM Xochimilco
  ✅  [   1090]  Iztacalco
  ✅  [   1134]  Hospital General de México
  ✅  [    677]  Tlahuac
  ✅  [   1809]  Chalco
  ✅  [   1830]  Montecillo
  ✅  [   2054]  Nezahualcóyotl

✅ 18 stations confirmed.


In [65]:
#  — Fetch real measurements
# For each station: get its sensors, then pull hourly data per sensor.
# This is the correct v3 pattern: location > sensors > hours.

HOURS_BACK = 8760
WHO_LIMITS = {'pm25': 15.0, 'pm10': 45.0, 'no2': 25.0,
              'o3': 100.0, 'co': 4000.0, 'so2': 40.0}

def get_sensors(loc_id):
    r = requests.get(f'{BASE_URL}/locations/{loc_id}/sensors',
                     headers=HEADERS, timeout=15)
    return r.json().get('results', []) if r.status_code == 200 else []

def fetch_hours(sensor_id, hours_back=None):
    date_from = datetime(2016, 1, 1, tzinfo=timezone.utc)
    date_to   = datetime(2016, 12, 31, tzinfo=timezone.utc)
    r = requests.get(
        f'{BASE_URL}/sensors/{sensor_id}/hours',
        headers=HEADERS,
        params={
            'datetime_from': date_from.strftime('%Y-%m-%dT%H:%M:%SZ'),
            'datetime_to':   date_to.strftime('%Y-%m-%dT%H:%M:%SZ'),
            'limit': 1000,
        },
        timeout=20
    )
    if r.status_code == 429:
        print(f'        ⏳ Rate limited, waiting 15s...')
        time.sleep(15)
        return fetch_hours(sensor_id)
    if r.status_code != 200:
        print(f'        ⚠️  sensor {sensor_id} HTTP {r.status_code}: {r.text[:120]}')
        return []
    return r.json().get('results', [])
def classify_diurnal(hour):
    if 7 <= hour <= 9 or 17 <= hour <= 19: return 'rush_hour'
    elif hour <= 5: return 'night'
    return 'daytime'


all_rows = []
failed   = []

for i, st in enumerate(STATIONS):
    print(f'[{i+1:02d}/{len(STATIONS)}] {st["name"][:45]:45s} ... ', end='', flush=True)

    sensors = get_sensors(st['id'])
    if not sensors:
        print('⚠️  no sensors')
        failed.append(st['name'])
        time.sleep(0.3)
        continue

    by_ts = {}  # timestamp → {pollutant: value}

    for sensor in sensors:
        sid   = sensor.get('id')
        param = sensor.get('parameter', {}).get('name', '').lower().replace('.','').replace(' ','')
        print(f'      sensor {sid}: param="{param}"')
        if param == 'pm2':   param = 'pm25'

        records = fetch_hours(sid, HOURS_BACK)
        for rec in records:
            ts    = rec.get('period', {}).get('datetimeFrom', {}).get('utc', '')
            value = rec.get('value')
            if ts and value is not None:
                by_ts.setdefault(ts, {})[param] = round(float(value), 3)
        time.sleep(1.2)

    if not by_ts:
        print('⚠️  no measurements')
        failed.append(st['name'])
        continue

    for ts, pollutants in by_ts.items():
        try:
            dt      = datetime.fromisoformat(ts.replace('Z', '+00:00'))
            hour    = dt.hour
            weekday = dt.weekday()
        except Exception:
            continue

        pm25 = pollutants.get('pm25')
        pm10 = pollutants.get('pm10')
        no2  = pollutants.get('no2')
        o3   = pollutants.get('o3')
        co   = pollutants.get('co')
        so2  = pollutants.get('so2')

        all_rows.append({
            'timestamp_utc':       ts,
            'station_id':          f"{st['country']}{str(st['id'])[-4:]}",
            'station_name':        st['name'],
            'city':                st['city'],
            'country_code':        st['country'],
            'latitude':            st['latitude'],
            'longitude':           st['longitude'],
            'sensor_type':         st['sensor_type'],
            'quadrant':            st['quadrant'],
            'openaq_location_id':  st['id'],
            'pm25_ugm3':           pm25,
            'pm10_ugm3':           pm10,
            'no2_ugm3':            no2,
            'o3_ugm3':             o3,
            'co_ugm3':             co,
            'so2_ugm3':            so2,
            'who_pm25_exceeded':   int(pm25 > WHO_LIMITS['pm25']) if pm25 is not None else None,
            'who_pm10_exceeded':   int(pm10 > WHO_LIMITS['pm10']) if pm10 is not None else None,
            'who_no2_exceeded':    int(no2  > WHO_LIMITS['no2'])  if no2  is not None else None,
            'diurnal_period':      classify_diurnal(hour),
            'is_weekend':          int(weekday >= 5),
        })

    print(f'✅  {len(by_ts)} hour-records')
    time.sleep(3.0)

print(f'\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'Total rows collected : {len(all_rows)}')
if failed:
    print(f'Stations skipped     : {failed}')

[01/18] Atizapán                                      ...       sensor 1299: param="so2"
      sensor 2350: param="pm10"
      sensor 2348: param="no2"
      sensor 2347: param="co"
      sensor 2349: param="o3"
✅  1182 hour-records
[02/18] Tultitlán                                     ...       sensor 1696: param="co"
      sensor 2864: param="o3"
      sensor 2865: param="no2"
      sensor 2863: param="pm10"
      sensor 1685: param="so2"
✅  1411 hour-records
[03/18] La Presa                                      ...       sensor 2278: param="co"
      sensor 2280: param="so2"
      sensor 2279: param="o3"
✅  1693 hour-records
[04/18] Xalostoc                                      ...       sensor 2448: param="pm25"
      sensor 2451: param="o3"
      sensor 2453: param="co"
      sensor 2450: param="so2"
      sensor 2452: param="no2"
      sensor 2449: param="pm10"
✅  1059 hour-records
[05/18] Villa de las Flores                           ...       sensor 2661: param="o3"
      senso

In [66]:
# 6 — Build DataFrame + z-score anomaly detection

df = pd.DataFrame(all_rows)
df = df.sort_values(['timestamp_utc','station_id']).reset_index(drop=True)

pollutant_cols = ['pm25_ugm3','pm10_ugm3','no2_ugm3','o3_ugm3','co_ugm3','so2_ugm3']

def flag_anomalies(df, col, z_thresh=3.0):
    flags = pd.Series('none', index=df.index)
    for sid, grp in df.groupby('station_id'):
        vals = pd.to_numeric(grp[col], errors='coerce')
        std  = vals.std()
        if pd.isna(std) or std == 0:
            flags.loc[grp.index] = 'data_freeze'
            continue
        z = (vals - vals.mean()) / std
        flags.loc[grp.index[z >  z_thresh]] = 'spike'
        flags.loc[grp.index[z < -z_thresh]] = 'sensor_dropout_suspected'
    return flags

primary = 'pm25_ugm3' if df['pm25_ugm3'].notna().any() else 'pm10_ugm3'
df['anomaly_type'] = flag_anomalies(df, primary)

all_missing = df[pollutant_cols].isnull().all(axis=1)
df.loc[all_missing, 'anomaly_type'] = 'communication_loss'

label_map = {'none':0,'spike':1,'data_freeze':2,'sensor_dropout_suspected':3,'communication_loss':4}
flag_map  = {'none':'VALID','spike':'ANOMALOUS_SPIKE','data_freeze':'FROZEN',
             'sensor_dropout_suspected':'DROPOUT_SUSPECTED','communication_loss':'MISSING'}

df['anomaly_label']     = df['anomaly_type'].map(label_map).fillna(0).astype(int)
df['data_quality_flag'] = df['anomaly_type'].map(flag_map).fillna('VALID')

print(f'✅ Shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'   Stations  : {df["station_id"].nunique()}')
print(f'   Countries : {df["country_code"].nunique()}')
print(f'\nAnomaly breakdown:')
print(df['data_quality_flag'].value_counts().to_string())
display(df.head(3))

✅ Shape: 23613 rows × 24 columns
   Stations  : 16
   Countries : 1

Anomaly breakdown:
data_quality_flag
FROZEN             16107
VALID               7464
ANOMALOUS_SPIKE       42


,timestamp_utc,station_id,station_name,city,country_code,latitude,longitude,sensor_type,quadrant,openaq_location_id,...,co_ugm3,so2_ugm3,who_pm25_exceeded,who_pm10_exceeded,who_no2_exceeded,diurnal_period,is_weekend,anomaly_type,anomaly_label,data_quality_flag
0,2016-03-10T06:00:00Z,MX1134,Hospital General de México,Mexico City,MX,19.411667,-99.152222,government,SW_UrbanCore,1134,...,NaN,NaN,NaN,0.0,0.0,daytime,0,none,0,VALID
1,2016-03-10T07:00:00Z,MX1090,Iztacalco,Mexico City,MX,19.384400,-99.117600,government,SW_UrbanCore,1090,...,0.5,0.001,NaN,NaN,0.0,rush_hour,0,data_freeze,2,FROZEN
2,2016-03-10T07:00:00Z,MX1134,Hospital General de México,Mexico City,MX,19.411667,-99.152222,government,SW_UrbanCore,1134,...,NaN,NaN,NaN,0.0,0.0,rush_hour,0,none,0,VALID


#clean up


In [68]:
# 7 - Cleanup cell
## It's important to know that while this cleanup cell is specific to the current dataset, should parameters be
### changed, it's amply recommended to visualize the data and do the clean-up accordingly to best suit individual needs.

# Fix 1: drop duplicates
before = len(df)
df = df.drop_duplicates(subset=['timestamp_utc', 'station_id'])
print(f'Duplicates removed : {before - len(df)} rows')

# Fix 2: rename units columns to reflect actual units
# NO2, O3, CO, SO2 are in ppm — rename columns accordingly
df = df.rename(columns={
    'no2_ugm3':  'no2_ppm',
    'o3_ugm3':   'o3_ppm',
    'co_ugm3':   'co_ppm',
    'so2_ugm3':  'so2_ppm',
})
# PM2.5 and PM10 stay in µg/m³ — those values look correct
print('Units columns renamed to reflect actual reported units.')

# Fix 3: re-run anomaly detection with minimum std threshold
# Stations with std < 0.5 (nearly constant) are not truly frozen
poll_primary = 'pm10_ugm3'  # most complete column at 42%

def flag_anomalies_clean(df, col, z_thresh=3.0, min_std=0.5):
    flags = pd.Series('none', index=df.index)
    for sid, grp in df.groupby('station_id'):
        vals = pd.to_numeric(grp[col], errors='coerce')
        std  = vals.std()
        if pd.isna(std) or std < min_std:
            continue  # not enough variance to judge — leave as 'none'
        z = (vals - vals.mean()) / std
        flags.loc[grp.index[z >  z_thresh]] = 'spike'
        flags.loc[grp.index[z < -z_thresh]] = 'sensor_dropout_suspected'
    return flags

df['anomaly_type'] = flag_anomalies_clean(df, poll_primary)
all_missing = df[['pm25_ugm3','pm10_ugm3','no2_ppm','o3_ppm','co_ppm','so2_ppm']].isnull().all(axis=1)
df.loc[all_missing, 'anomaly_type'] = 'communication_loss'

label_map = {'none':0,'spike':1,'sensor_dropout_suspected':2,'communication_loss':3}
flag_map  = {'none':'VALID','spike':'ANOMALOUS_SPIKE',
             'sensor_dropout_suspected':'DROPOUT_SUSPECTED','communication_loss':'MISSING'}
df['anomaly_label']     = df['anomaly_type'].map(label_map).fillna(0).astype(int)
df['data_quality_flag'] = df['anomaly_type'].map(flag_map).fillna('VALID')

print(f'\nFinal shape : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\nAnomaly breakdown:')
print(df['data_quality_flag'].value_counts().to_string())

Duplicates removed : 2144 rows
Units columns renamed to reflect actual reported units.

Final shape : 21,469 rows × 24 columns

Anomaly breakdown:
data_quality_flag
VALID              21367
ANOMALOUS_SPIKE      102


In [69]:
# 7 — Save and download
from google.colab import files

filename = 'openaq_autonomic_monitoring_dataset.csv' # change this for your .csv file title name
df.to_csv(filename, index=False)
print(f'✅ Saved: {filename}  ({len(df)} rows × {len(df.columns)} columns)')
files.download(filename)


✅ Saved: openaq_autonomic_monitoring_dataset.csv  (21469 rows × 24 columns)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Download started — upload CSV + README.txt to Zenodo.
